# songo-model-stockfish-for-google-collab - All In One

Notebook unique pour:
- monter Google Drive
- cloner ou mettre a jour le code depuis GitHub
- installer les dependances
- lancer la generation de dataset
- construire le dataset final
- entrainer le modele
- evaluer le modele
- benchmarker le modele
- reprendre un job interrompu avec le meme `job_id`


## 1. Monter Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuration du projet

In [ ]:
GIT_REPO_URL = 'https://github.com/TON-COMPTE/songo-model-stockfish-for-google-collab.git'
GIT_BRANCH = 'main'
PROJECT_NAME = 'songo-model-stockfish-for-google-collab'
DRIVE_ROOT = '/content/drive/MyDrive/songo-stockfish'
WORKTREE = f'/content/{PROJECT_NAME}'

print('GIT_REPO_URL =', GIT_REPO_URL)
print('GIT_BRANCH   =', GIT_BRANCH)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('WORKTREE     =', WORKTREE)


## 3. Initialiser Drive

In [ ]:
import os

for path in [
    f'{DRIVE_ROOT}/code',
    f'{DRIVE_ROOT}/code_snapshots',
    f'{DRIVE_ROOT}/data/raw',
    f'{DRIVE_ROOT}/data/raw_colab_pro',
    f'{DRIVE_ROOT}/data/raw_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/sampled',
    f'{DRIVE_ROOT}/data/sampled_colab_pro',
    f'{DRIVE_ROOT}/data/sampled_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/datasets',
    f'{DRIVE_ROOT}/jobs',
    f'{DRIVE_ROOT}/models/checkpoints',
    f'{DRIVE_ROOT}/models/final',
    f'{DRIVE_ROOT}/models/promoted/best',
    f'{DRIVE_ROOT}/models/lineage',
    f'{DRIVE_ROOT}/logs',
    f'{DRIVE_ROOT}/reports/evaluations',
    f'{DRIVE_ROOT}/reports/benchmarks',
    f'{DRIVE_ROOT}/exports',
]:
    os.makedirs(path, exist_ok=True)

print('Drive layout ready')


## 4. Cloner ou mettre a jour le repo

In [ ]:
import os
import shutil
import subprocess

if not os.path.exists(os.path.join(WORKTREE, '.git')):
    if os.path.exists(WORKTREE):
        shutil.rmtree(WORKTREE)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, WORKTREE], check=True)
else:
    subprocess.run(['git', '-C', WORKTREE, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'pull', '--ff-only', 'origin', GIT_BRANCH], check=True)

print('Worktree ready')


## 5. Installer les dependances

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', f'{WORKTREE}/requirements.txt'], check=True)
print('Environment ready in Colab runtime')


## 6. Choisir les configs et job ids

In [ ]:
DATASET_GENERATE_CONFIG = 'config/dataset_generation.full_matrix.colab_pro.yaml'
DATASET_BUILD_CONFIG = 'config/dataset_build.colab_pro.yaml'
TRAIN_CONFIG = 'config/train.colab_pro.yaml'
EVALUATION_CONFIG = 'config/evaluation.colab_pro.yaml'
BENCHMARK_CONFIG = 'config/benchmark.colab_pro.yaml'

DATASET_GENERATE_JOB_ID = 'dataset_full_matrix_colab_pro_v1'
DATASET_BUILD_JOB_ID = 'dataset_build_full_matrix_colab_pro_v1'
TRAIN_JOB_ID = 'train_colab_pro_cycle_001'
EVALUATION_JOB_ID = 'eval_colab_pro_cycle_001'
BENCHMARK_JOB_ID = 'benchmark_colab_pro_cycle_001'


## 7. Lancer la generation du dataset

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-generate --config $DATASET_GENERATE_CONFIG --job-id $DATASET_GENERATE_JOB_ID"

## 8. Construire le dataset final

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-build --config $DATASET_BUILD_CONFIG --job-id $DATASET_BUILD_JOB_ID"

## 9. Entrainement GPU

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main train --config $TRAIN_CONFIG --job-id $TRAIN_JOB_ID"

## 10. Evaluation

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main evaluate --config $EVALUATION_CONFIG --job-id $EVALUATION_JOB_ID"

## 11. Benchmark

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main benchmark --config $BENCHMARK_CONFIG --job-id $BENCHMARK_JOB_ID"

## 12. Suivre un job en direct

In [ ]:
WATCH_JOB_ID = TRAIN_JOB_ID
!tail -f {DRIVE_ROOT}/jobs/{WATCH_JOB_ID}/events.jsonl

## 13. Reprendre un job interrompu

Relance simplement la meme cellule de commande avec le meme `job_id`.